# torch.export Deep Dive — Advanced Export Techniques

**Module 37** | Prerequisites: Module 08 (torch.compile), Module 11 (Export & Deployment) | Time: ~3 hours

Module 11 covered the basics. This notebook goes deeper: ExportedProgram anatomy, control flow primitives, custom ops, the two IR levels, and practical debugging.

---

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.export import Dim, export
import tempfile
from pathlib import Path

print(f"PyTorch: {torch.__version__}")
print(f"Device: CPU (all examples run on CPU)")

## 1. ExportedProgram Anatomy

An `ExportedProgram` contains everything needed to run a model without the original Python source: the FX graph, state dict, constraints, and metadata.

In [ ]:
class SimpleModel(nn.Module):
    def __init__(self):
        super().__init__()
        self.linear = nn.Linear(10, 5)
        self.register_buffer("scale", torch.tensor(2.0))

    def forward(self, x):
        return self.linear(x) * self.scale

model = SimpleModel()
ep = export(model, (torch.randn(3, 10),))

print(f"Type: {type(ep).__name__}")
print(f"State dict keys: {list(ep.state_dict.keys())}")
print(f"Range constraints: {ep.range_constraints}")
print(f"Constants: {ep.constants}")
# Inspect the FX graph
ep.graph_module.graph.print_tabular()

# Verify correctness
result_orig = model(torch.randn(3, 10))
result_export = ep.module()(torch.randn(3, 10))
print(f"\nGraph module type: {type(ep.graph_module).__name__}")

## 2. Graph Signature Inspection

The graph signature tells you what each graph input/output represents — parameter, buffer, or user data.

In [ ]:
class ModelWithBuffer(nn.Module):
    def __init__(self):
        super().__init__()
        self.linear = nn.Linear(8, 4)
        self.register_buffer("running_mean", torch.zeros(4))

    def forward(self, x):
        out = self.linear(x)
        self.running_mean = self.running_mean * 0.9 + out.mean(dim=0) * 0.1
        return out

model_buf = ModelWithBuffer()
ep_buf = export(model_buf, (torch.randn(2, 8),))

print("Input Specs:")
for spec in ep_buf.graph_signature.input_specs:
    target = spec.target if spec.target else "N/A"
    print(f"  {spec.kind}: {spec.arg.name} -> {target}")

print("\nOutput Specs:")
for spec in ep_buf.graph_signature.output_specs:
    target = spec.target if spec.target else "N/A"
    print(f"  {spec.kind}: {spec.arg.name} -> {target}")

## 3. Dynamic Shapes with Dim API

The `Dim` API gives fine-grained control over which dimensions are dynamic and their valid ranges.

In [ ]:
class DynModel(nn.Module):
    def __init__(self):
        super().__init__()
        self.linear = nn.Linear(16, 8)

    def forward(self, x):
        return self.linear(x)

dyn_model = DynModel()

# Single dynamic dim with bounds
batch = Dim("batch", min=1, max=128)
ep_dyn = export(
    dyn_model,
    (torch.randn(4, 16),),
    dynamic_shapes={"x": {0: batch}},
)

print(f"Constraints: {ep_dyn.range_constraints}")

for bs in [1, 8, 32, 64]:
    result = ep_dyn.module()(torch.randn(bs, 16))
    print(f"  batch={bs}: output shape = {result.shape}")

In [ ]:
# Shared dims: two inputs must have the same batch size
class TwoInputModel(nn.Module):
    def forward(self, x, y):
        return x + y

shared_batch = Dim("batch", min=1, max=64)
ep_shared = export(
    TwoInputModel(),
    (torch.randn(4, 10), torch.randn(4, 10)),
    dynamic_shapes={"x": {0: shared_batch}, "y": {0: shared_batch}},
)

result = ep_shared.module()(torch.randn(16, 10), torch.randn(16, 10))
print(f"Shared batch=16: output shape = {result.shape}")
# Dim.AUTO: let export infer the constraints
ep_auto = export(
    dyn_model,
    (torch.randn(4, 16),),
    dynamic_shapes={"x": {0: Dim.AUTO}},
)
print(f"AUTO constraints: {ep_auto.range_constraints}")

for bs in [1, 8, 32]:
    result = ep_auto.module()(torch.randn(bs, 16))
    print(f"  batch={bs}: shape = {result.shape}")

## 4. torch.cond for Conditional Control Flow

Python `if/else` only captures one branch. `torch.cond` captures both.

In [ ]:
# Problem: Python if/else fails or captures only one branch
class BadBranchModel(nn.Module):
    def forward(self, x):
        if x.sum() > 0:
            return x * 2
        else:
            return x * -1

try:
    ep_bad = export(BadBranchModel(), (torch.randn(4),))
    print("Exported, but only one branch captured — this is WRONG")
except Exception as e:
    print(f"Export failed (expected): {str(e).split(chr(10))[0]}")

In [ ]:
# Solution: torch.cond
class GoodBranchModel(nn.Module):
    def forward(self, x):
        return torch.cond(
            x.sum() > 0,
            lambda x: x * 2,
            lambda x: x * -1,
            (x,),
        )

model_cond = GoodBranchModel()
ep_cond = export(model_cond, (torch.randn(4),))
print("torch.cond export succeeded!")

# Verify both branches work
pos = torch.ones(4)
neg = -torch.ones(4)
print(f"Positive input: {ep_cond.module()(pos).tolist()}")
print(f"Negative input: {ep_cond.module()(neg).tolist()}")

In [ ]:
# torch.cond with multiple outputs
class MultiOutCond(nn.Module):
    def forward(self, x, y):
        def true_fn(x, y):
            return x + 1, y * 2

        def false_fn(x, y):
            return x - 1, y * 0.5

        return torch.cond(x.sum() > 0, true_fn, false_fn, (x, y))

ep_multi = export(MultiOutCond(), (torch.randn(3), torch.randn(3)))
a, b = ep_multi.module()(torch.ones(3), torch.ones(3))
print(f"True branch: a={a.tolist()}, b={b.tolist()}")

a, b = ep_multi.module()(-torch.ones(3), torch.ones(3))
print(f"False branch: a={a.tolist()}, b={b.tolist()}")

## 5. torch.while_loop for Loops

For data-dependent iteration counts, `torch.while_loop` provides exportable loops.

In [ ]:
class LoopModel(nn.Module):
    def forward(self, x, max_iter):
        def cond_fn(x, count, max_iter):
            return count < max_iter

        def body_fn(x, count, max_iter):
            return x * 0.9, count + 1, max_iter

        result_x, final_count, _ = torch.while_loop(
            cond_fn, body_fn, (x, torch.tensor(0), max_iter)
        )
        return result_x, final_count

loop_model = LoopModel()
x = torch.ones(5)
max_iter = torch.tensor(10)

ep_loop = export(loop_model, (x, max_iter))
result, count = ep_loop.module()(x, max_iter)

print(f"After {count.item():.0f} iterations:")
print(f"  x[0] = {result[0].item():.6f} (expected: {0.9**10:.6f})")
print(f"  Match: {abs(result[0].item() - 0.9**10) < 1e-6}")

## 6. draft_export for Debugging

When export fails, `draft_export` returns a report of all issues instead of crashing on the first one.

In [ ]:
from torch.export import draft_export

class ProblematicModel(nn.Module):
    def __init__(self):
        super().__init__()
        self.linear = nn.Linear(10, 5)

    def forward(self, x):
        if x.sum() > 0:
            return self.linear(x)
        else:
            return self.linear(x) * -1

prob_model = ProblematicModel()
args = (torch.randn(3, 10),)

# Strict export fails
try:
    export(prob_model, args)
except Exception as e:
    print(f"Strict export failed: {str(e).split(chr(10))[0]}")

# draft_export gives a report
try:
    ep_draft, report = draft_export(prob_model, args)
    print(f"\ndraft_export succeeded: {ep_draft is not None}")
    if report:
        print(f"Report: {report}")
except Exception as e:
    print(f"draft_export issue: {str(e).split(chr(10))[0]}")

## 7. Pre-Dispatch vs Post-Dispatch IR

Pre-dispatch preserves high-level ops (closer to user code). Post-dispatch decomposes them for backends.

In [ ]:
class IRModel(nn.Module):
    def __init__(self):
        super().__init__()
        self.linear = nn.Linear(10, 10)
        self.norm = nn.LayerNorm(10)

    def forward(self, x):
        return torch.relu(self.norm(self.linear(x)))

ir_model = IRModel()
ir_args = (torch.randn(4, 10),)

# Post-dispatch (default) — more ops, lower level
ep_post = export(ir_model, ir_args)
post_ops = sorted(set(
    str(n.target).split(".")[-1]
    for n in ep_post.graph_module.graph.nodes
    if n.op == "call_function"
))
print(f"Post-dispatch ({len(post_ops)} unique ops): {post_ops}")

# Pre-dispatch — fewer ops, higher level
ep_pre = export(ir_model, ir_args, pre_dispatch=True)
pre_ops = sorted(set(
    str(n.target).split(".")[-1]
    for n in ep_pre.graph_module.graph.nodes
    if n.op == "call_function"
))
print(f"Pre-dispatch ({len(pre_ops)} unique ops):  {pre_ops}")

In [ ]:
# Convert pre-dispatch to post-dispatch with run_decompositions()
ep_decomposed = ep_pre.run_decompositions()
decomp_ops = sorted(set(
    str(n.target).split(".")[-1]
    for n in ep_decomposed.graph_module.graph.nodes
    if n.op == "call_function"
))
print(f"After decomposition ({len(decomp_ops)} unique ops): {decomp_ops}")

# Verify results match
r_pre = ep_pre.module()(*ir_args)
r_post = ep_decomposed.module()(*ir_args)
print(f"Results match: {torch.allclose(r_pre, r_post, atol=1e-6)}")

## 8. Custom Ops in Export

Custom ops need `register_fake` so export can trace through them without running the real computation.

In [ ]:
# Define a custom op
@torch.library.custom_op("notebook_demo::leaky_relu_squared", mutates_args=())
def leaky_relu_squared(x: torch.Tensor, neg_slope: float) -> torch.Tensor:
    return torch.where(x > 0, x ** 2, neg_slope * x ** 2)

# Register fake implementation for export
@leaky_relu_squared.register_fake
def leaky_relu_squared_fake(x, neg_slope):
    return torch.empty_like(x)

class CustomOpModel(nn.Module):
    def forward(self, x):
        return torch.ops.notebook_demo.leaky_relu_squared(x, 0.01)

ep_custom = export(CustomOpModel(), (torch.randn(3, 4),))
print("Custom op export succeeded!")

x_test = torch.tensor([-2.0, -1.0, 0.0, 1.0, 2.0])
result = ep_custom.module()(x_test)
print(f"Input:  {x_test.tolist()}")
print(f"Output: {result.tolist()}")

## 9. Strict vs Non-Strict Export

Strict mode (default) catches all issues. Non-strict is more permissive for development.

In [ ]:
class ClampModel(nn.Module):
    def __init__(self):
        super().__init__()
        self.linear = nn.Linear(10, 5)
        self.threshold = 0.5

    def forward(self, x):
        out = self.linear(x)
        return torch.clamp(out, min=-self.threshold, max=self.threshold)

clamp_model = ClampModel()
clamp_args = (torch.randn(3, 10),)

# Strict
ep_strict = export(clamp_model, clamp_args, strict=True)
strict_nodes = sum(1 for n in ep_strict.graph_module.graph.nodes)
print(f"Strict: {strict_nodes} nodes")

# Non-strict
ep_nonstrict = export(clamp_model, clamp_args, strict=False)
nonstrict_nodes = sum(1 for n in ep_nonstrict.graph_module.graph.nodes)
print(f"Non-strict: {nonstrict_nodes} nodes")

# Both produce same results
r1 = ep_strict.module()(*clamp_args)
r2 = ep_nonstrict.module()(*clamp_args)
print(f"Same results: {torch.allclose(r1, r2)}")

## 10. Practical Debugging Workflow

The complete flow: try export → fail → draft_export → fix → retry → save.

In [ ]:
# Step 1: Model with issues
class DebugModel(nn.Module):
    def __init__(self):
        super().__init__()
        self.pos_path = nn.Linear(10, 5)
        self.neg_path = nn.Linear(10, 5)

    def forward(self, x):
        # This uses data-dependent control flow — won't export!
        if x.sum() > 0:
            return self.pos_path(x)
        return self.neg_path(x)

debug_model = DebugModel()
debug_args = (torch.randn(3, 10),)

print("Step 1: Try export...")
try:
    export(debug_model, debug_args)
    print("  Succeeded (only one branch captured)")
except Exception as e:
    print(f"  Failed: {str(e).split(chr(10))[0]}")

In [ ]:
# Step 2: Fix with torch.cond
class FixedModel(nn.Module):
    def __init__(self):
        super().__init__()
        self.pos_path = nn.Linear(10, 5)
        self.neg_path = nn.Linear(10, 5)

    def forward(self, x):
        def true_fn(x, pw, pb, nw, nb):
            return F.linear(x, pw, pb)

        def false_fn(x, pw, pb, nw, nb):
            return F.linear(x, nw, nb)

        return torch.cond(
            x.sum() > 0,
            true_fn,
            false_fn,
            (x, self.pos_path.weight, self.pos_path.bias,
             self.neg_path.weight, self.neg_path.bias),
        )

fixed_model = FixedModel()
# Copy weights from original
fixed_model.pos_path.load_state_dict(debug_model.pos_path.state_dict())
fixed_model.neg_path.load_state_dict(debug_model.neg_path.state_dict())

print("Step 2: Export fixed model...")
ep_fixed = export(fixed_model, debug_args)
print("  Success!")

In [ ]:
# Step 3: Validate
print("Step 3: Validate...")
n_correct = 0
for _ in range(50):
    test_x = torch.randn(3, 10)
    orig = debug_model(test_x)
    fixed = ep_fixed.module()(test_x)
    if torch.allclose(orig, fixed, atol=1e-6):
        n_correct += 1
print(f"  {n_correct}/50 random inputs match")
# Step 4: Save and load
print("Step 4: Save and load...")
with tempfile.TemporaryDirectory() as tmpdir:
    path = Path(tmpdir) / "fixed_model.pt2"
    torch.export.save(ep_fixed, str(path))
    print(f"  Saved: {path.stat().st_size:,} bytes")

    ep_loaded = torch.export.load(str(path))
    result = ep_loaded.module()(torch.randn(3, 10))
    print(f"  Loaded and ran: output shape = {result.shape}")
    print("  Round-trip complete!")

## 11. Retraceability

An exported program can be re-exported with different constraints.

In [ ]:
retrace_model = DynModel()
retrace_args = (torch.randn(4, 16),)

# First export: wide constraints
batch_wide = Dim("batch", min=1, max=256)
ep1 = export(retrace_model, retrace_args, dynamic_shapes={"x": {0: batch_wide}})
print(f"First export constraints:  {ep1.range_constraints}")

# Re-export: tighter constraints
batch_tight = Dim("batch", min=1, max=32)
ep2 = export(ep1.module(), retrace_args, dynamic_shapes={"x": {0: batch_tight}})
print(f"Re-exported constraints: {ep2.range_constraints}")

# Both work
r1 = ep1.module()(torch.randn(16, 16))
r2 = ep2.module()(torch.randn(16, 16))
print(f"Both produce outputs: {r1.shape}, {r2.shape}")

---

## Exercise

**Task**: Export a model with a conditional — if the sum of the input exceeds a threshold, apply ReLU; otherwise, apply Sigmoid. Use `torch.cond`.

Requirements:
1. Define a model with a `nn.Linear` layer followed by a conditional activation
2. Use `torch.cond` for the conditional
3. Export with a dynamic batch dimension
4. Verify both branches produce correct results
5. Save and load the exported program

In [ ]:
# Your solution here
class ConditionalActivation(nn.Module):
    def __init__(self, in_features, out_features, threshold=0.0):
        super().__init__()
        self.linear = nn.Linear(in_features, out_features)
        self.threshold = threshold

    def forward(self, x):
        out = self.linear(x)
        # TODO: use torch.cond to apply relu if out.sum() > threshold,
        # else apply sigmoid
        # Hint: the predicate must be a tensor comparison
        pass  # Replace with your implementation

# Test your implementation:
# 1. Create the model
# 2. Export with dynamic batch dim
# 3. Test with positive-sum and negative-sum inputs
# 4. Save and load
print("Implement the exercise above!")

In [ ]:
# Reference solution
class ConditionalActivationSolution(nn.Module):
    def __init__(self, in_features, out_features, threshold=0.0):
        super().__init__()
        self.linear = nn.Linear(in_features, out_features)
        self.threshold = threshold

    def forward(self, x):
        out = self.linear(x)
        return torch.cond(
            out.sum() > self.threshold,
            lambda t: torch.relu(t),
            lambda t: torch.sigmoid(t),
            (out,),
        )

sol_model = ConditionalActivationSolution(10, 5, threshold=0.0)
batch_dim = Dim("batch", min=1, max=64)
ep_sol = export(
    sol_model,
    (torch.randn(4, 10),),
    dynamic_shapes={"x": {0: batch_dim}},
)
print(f"Export succeeded with constraints: {ep_sol.range_constraints}")

# Verify
for _ in range(20):
    test = torch.randn(8, 10)
    assert torch.allclose(sol_model(test), ep_sol.module()(test), atol=1e-6)
print("All 20 random tests passed!")

# Save/load round-trip
with tempfile.TemporaryDirectory() as tmpdir:
    p = Path(tmpdir) / "conditional.pt2"
    torch.export.save(ep_sol, str(p))
    ep_rt = torch.export.load(str(p))
    test = torch.randn(2, 10)
    assert torch.allclose(ep_sol.module()(test), ep_rt.module()(test))
    print(f"Round-trip verified ({p.stat().st_size:,} bytes)")

---

## Key Takeaways

1. **ExportedProgram** bundles graph, weights, constraints, and metadata — fully self-contained
2. **Graph signature** categorizes every input (parameter, buffer, user input) and output
3. **Dim API** gives fine-grained dynamic shape control; shared dims enforce cross-input constraints
4. **torch.cond** replaces Python `if/else` — both branches are traced and preserved in the graph
5. **torch.while_loop** handles data-dependent iteration without unrolling
6. **draft_export** reports all issues instead of failing on the first — essential for debugging
7. **Pre-dispatch IR** preserves high-level ops; **post-dispatch** decomposes for backends
8. **Custom ops** need `register_fake` for export to trace through them
9. **Strict mode** is the gold standard for production; **non-strict** is useful for iterative development
10. **PT2 archive** serialization enables save/load with full backward compatibility

---

*Module 37 of the PyTorch Complete Learning Guide*